# 08 - Deep learning model 1: MLP with embeddings (PyTorch)
Needs notebook 05 output and `pip install torch`. Early stopping and the threshold use the validation set only.

In [1]:
import sys, os
from pathlib import Path
# works whether Jupyter starts in the project root or inside notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))      # lets us import data_utils.py from the root
print("Project root:", ROOT)

Project root: c:\Users\aru12\OneDrive\Desktop\capstone project


## 1. Imports, data and category vocabularies (train only)

In [3]:
%pip install torch

  Using cached torch-2.14.1-cp311-cp311-win_amd64.whl.metadata (38 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.9.0-py3-none-any.whl.metadata (10 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
Using cached torch-2.14.1-cp311-cp311-win_amd64.whl (124.1 MB)
Using cached fsspec-2026.9.0-py3-none-any.whl (221 kB)
Using cached networkx-3.6.1-py3-none-any.whl (2.1 MB)
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)

   ---------------------------------------- 0/7 [mpmath]
   ---------------------------------------- 0/7 [mpmath]
   ---------------------------------------- 0/7 [mpmath]
   ----- ---------------------------------- 1/7 [sympy]
   ----- ---------------------------------- 1/7 [sympy]

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [4]:
import copy, numpy as np, pandas as pd, torch, torch.nn as nn
from sklearn.metrics import roc_auc_score
from data_utils import *

torch.manual_seed(SEED); np.random.seed(SEED)
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", dev)
(Xtr, ytr), (Xva, yva), (Xte, yte) = load_splits()

# vocabularies come from TRAIN only; unseen categories map to index 0
vocab = {c: {v: i + 1 for i, v in enumerate(sorted(Xtr[c].unique()))} for c in FEATURES}
def enc(X):
    return torch.tensor(np.stack([X[c].map(vocab[c]).fillna(0).astype(int).to_numpy()
                                  for c in FEATURES], 1), dtype=torch.long).to(dev)
Ctr, Cva, Cte = enc(Xtr), enc(Xva), enc(Xte)
Ytr = torch.tensor(ytr, dtype=torch.float32).to(dev)

device: cpu


## 2. Model, loss (class-weighted) and optimizer

In [5]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.embs = nn.ModuleList([nn.Embedding(len(vocab[c]) + 1, min(32, len(vocab[c]) // 2 + 2))
                                   for c in FEATURES])
        d = sum(e.embedding_dim for e in self.embs)
        self.mlp = nn.Sequential(nn.Linear(d, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
                                 nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.3),
                                 nn.Linear(64, 1))
    def forward(self, c):
        return self.mlp(torch.cat([e(c[:, i]) for i, e in enumerate(self.embs)], 1)).squeeze(1)

net = Net().to(dev)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor((ytr == 0).sum() / (ytr == 1).sum(),
                                                        dtype=torch.float32).to(dev))
opt = torch.optim.AdamW(net.parameters(), lr=2e-3, weight_decay=1e-4)
print(net)

Net(
  (embs): ModuleList(
    (0): Embedding(32, 17)
    (1): Embedding(139, 32)
    (2): Embedding(276, 32)
  )
  (mlp): Sequential(
    (0): Linear(in_features=81, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=64, out_features=1, bias=True)
  )
)


## 3. Train with early stopping on validation AUC

In [6]:
def predict(C):
    net.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(C), 8192):
            out.append(torch.sigmoid(net(C[i:i + 8192])))
    return torch.cat(out).cpu().numpy()

best_auc, best_state, bad, BS = 0, None, 0, 2048
for ep in range(30):
    net.train(); perm = torch.randperm(len(Ctr), device=dev)
    for i in range(0, len(perm), BS):
        idx = perm[i:i + BS]
        opt.zero_grad(); loss_fn(net(Ctr[idx]), Ytr[idx]).backward(); opt.step()
    auc = roc_auc_score(yva, predict(Cva))            # early stopping uses VALIDATION only
    print(f"epoch {ep+1:02d}  val_auc={auc:.4f}")
    if auc > best_auc: best_auc, best_state, bad = auc, copy.deepcopy(net.state_dict()), 0
    else:
        bad += 1
        if bad >= 4: break

epoch 01  val_auc=0.9149
epoch 02  val_auc=0.9163
epoch 03  val_auc=0.9175
epoch 04  val_auc=0.9175
epoch 05  val_auc=0.9179
epoch 06  val_auc=0.9183
epoch 07  val_auc=0.9180
epoch 08  val_auc=0.9184
epoch 09  val_auc=0.9182
epoch 10  val_auc=0.9182
epoch 11  val_auc=0.9186
epoch 12  val_auc=0.9182
epoch 13  val_auc=0.9184
epoch 14  val_auc=0.9186
epoch 15  val_auc=0.9185


## 4. Evaluate on test (threshold from validation) and save the result

In [7]:
net.load_state_dict(best_state)
t = best_threshold(yva, predict(Cva))
p_te = predict(Cte)
report("MLP (embeddings)", yte, p_te, t)
print("test ROC-AUC:", round(roc_auc_score(yte, p_te), 4))
save_rows([metrics_row("MLP (embeddings)", yte, p_te, t)])
pd.read_csv(RESULTS)

MLP (embeddings)       thr=0.57  accuracy=0.8327  balanced_acc=0.8336  gap=0.0009
test ROC-AUC: 0.9136


,Model,Features,Train/Val/Test,Test_rows,Threshold,Accuracy,Balanced_Accuracy,Precision,Recall,F1,ROC_AUC
0,Wide&Deep (DL),Primary Type + Location Description + Beat,2023-Jun24 / Jul-Dec24 / 2025,236927,0.58,0.8332,0.8320,0.5408,0.8301,0.6549,0.9127
1,LogisticRegression,Primary Type + Location Description + Beat,2023-Jun24 / Jul-Dec24 / 2025,236927,0.53,0.8271,0.8276,0.5299,0.8285,0.6463,0.9094
2,RandomForest,Primary Type + Location Description + Beat,2023-Jun24 / Jul-Dec24 / 2025,236927,0.51,0.8304,0.8301,0.5357,0.8295,0.6510,0.9104
3,HistGradientBoosting,Primary Type + Location Description + Beat,2023-Jun24 / Jul-Dec24 / 2025,236927,0.55,0.8296,0.8299,0.5342,0.8303,0.6501,0.9113
4,XGBoost,Primary Type + Location Description + Beat,2023-Jun24 / Jul-Dec24 / 2025,236927,0.52,0.8312,0.8311,0.5371,0.8309,0.6524,0.9125
5,LightGBM,Primary Type + Location Description + Beat,2023-Jun24 / Jul-Dec24 / 2025,236927,0.55,0.8302,0.8301,0.5353,0.8299,0.6508,0.9114
6,MLP (embeddings),Primary Type + Location Description + Beat,2023-Jun24 / Jul-Dec24 / 2025,236927,0.57,0.8327,0.8336,0.5397,0.8350,0.6556,0.9136
